In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample, shuffle
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import random

# Define content path
content_path = "../data/"

/home/azureuser/NLP_CW/nlp_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# making results determinsistic
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)  # Hugging Face helper

## Data Loading and Preprocessing

In [3]:
# Load the dataset and preprocess it for training.

print("--- Loading and Preprocessing Data ---")
# Full data
df_full = pd.read_csv(
    content_path+"dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=4,
    header=None
)
df_full.columns = ["par_id", "art_id", "keyword", "country_code", "text", "label"]

# Train data
df_train_raw = pd.read_csv(content_path+"train_semeval_parids-labels.csv")
train_df  = df_full.merge(
    df_train_raw.drop(columns=['label']),
    on='par_id', how='inner'
)
train_df = train_df[['par_id', 'art_id', 'keyword', 'country_code','text', 'label']].dropna(subset=['text'])
train_df['label'] = train_df['label'].apply(lambda x: 0 if x in [0, 1] else 1)

# Test Data
df_test_raw = pd.read_csv(content_path+"dev_semeval_parids-labels.csv")
# creating dataset for training classifier
df_cls_test_cols = ['text', 'label', 'par_id', 'keyword', 'country_code']
test_df = df_full.merge(
    df_test_raw.drop(columns=['label']),  # drop label to avoid confusion during merge
    on='par_id', how='inner'
    )
test_df = test_df[df_cls_test_cols]
test_df['text'] = test_df['text'].fillna('')
test_df['label'] = test_df['label'].apply(lambda x: 0 if x in [0, 1] else 1)
test_df.head(2)

## Data Splitting
# Split the data into training and validation sets.

print("--- Splitting the Raw Training Data ---")
internal_train_df, internal_val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=1
)

--- Loading and Preprocessing Data ---
--- Splitting the Raw Training Data ---


In [4]:
# Convert pandas DataFrames to Hugging Face Datasets.

print("--- Converting to Hugging Face Datasets ---")
train_dataset = Dataset.from_pandas(internal_train_df)
val_dataset = Dataset.from_pandas(internal_val_df)

--- Converting to Hugging Face Datasets ---


## Model and Tokenizer

In [5]:
# Load the RoBERTa-base model and tokenizer.

print("--- Loading RoBERTa-base Model and Tokenizer ---")
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

## Model Initialization
def model_init():
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    return model

--- Loading RoBERTa-base Model and Tokenizer ---


Map: 100%|██████████| 1675/1675 [00:00<00:00, 12652.44 examples/s]


## Weighting the classes

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

labels = np.array(tokenized_train["label"])
class_counts = np.bincount(labels)
weights = 1.0 / np.sqrt(class_counts)
weights = weights / weights.sum() * len(weights)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
class_weights

tensor([0.4889, 1.5111], device='cuda:0')

## Custom Trainer for Class Weighting

In [7]:
# Create a custom Trainer to incorporate class weights into the loss function.

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary', zero_division=0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

## Training

In [8]:
# Configure and run the training process.

print("--- Training The Model with Class Weighting---")

learning_rate = 2e-5
weight_decay = 0.01
EPOCHS = 4

training_args = TrainingArguments(
    output_dir="./class_weight_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to=[]
)

trainer = CustomTrainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print(f"Training for {EPOCHS} epochs with class weighting...")
trainer.train()

# Save the trained model and tokenizer.

model_path = "model/"
trainer.save_model(model_path+"class_weighting_model")
tokenizer.save_pretrained(model_path+"class_weighting_tokenizer")
print(f"Final model saved to {model_path}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- Training The Model with Class Weighting---


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training for 4 epochs with class weighting...


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.437300,0.362107,0.512397,0.455882,0.584906
2,0.289300,0.335778,0.509960,0.373178,0.805031
3,0.204800,0.440328,0.548023,0.497436,0.610063
4,0.123700,0.527746,0.571429,0.542373,0.603774


Final model saved to model/


## Prediction on official dev set

In [10]:
# --- Load Test Data ---

print("--- Loading Test Data ---")
dev_df = pd.read_csv(content_path+"dev_semeval_parids-labels.csv")

official_dev_df = df_full.merge(
    dev_df.drop(columns=['label']),
    on='par_id', how='inner'
)

official_dev_df = official_dev_df[['text', 'keyword', 'country_code','label', 'par_id']]
official_dev_df['text'] = official_dev_df['text'].fillna('')
official_dev_df['label'] = official_dev_df['label'].apply(lambda x: 0 if x in [0, 1] else 1)
official_dev_dataset = Dataset.from_pandas(official_dev_df)
official_tokenized_dev = official_dev_dataset.map(tokenize_function, batched=True)

print("\n--- Generating Predictions on Test Set ---")
# predicting on tokenized_test
official_dev_test_outputs = trainer.predict(test_dataset=official_tokenized_dev)

# Extract the raw logits
official_dev_raw_logits = official_dev_test_outputs.predictions

# Convert logits to final predicted classes (0 or 1) using argmax
# This looks at the two scores for each row and picks the index of the higher one
official_dev_predicted_classes = np.argmax(official_dev_raw_logits, axis=1)

# Attach the predictions back tp original pandas DataFrame
official_dev_df['predicted_label'] = official_dev_predicted_classes

print("\n--- Final Model Performance on Test Set ---")
print(f"Test F1 Score: {official_dev_test_outputs.metrics['test_f1']:.4f}")
print(f"Test Precision: {official_dev_test_outputs.metrics['test_precision']:.4f}")
print(f"Test Recall: {official_dev_test_outputs.metrics['test_recall']:.4f}")

# 5. Check out your new DataFrame!
print("\n--- Sample of Test Data with Predictions ---")
official_dev_df.head()

--- Loading Test Data ---


Map: 100%|██████████| 2094/2094 [00:00<00:00, 12935.47 examples/s]


--- Generating Predictions on Test Set ---



--- Final Model Performance on Test Set ---
Test F1 Score: 0.6047
Test Precision: 0.5628
Test Recall: 0.6533

--- Sample of Test Data with Predictions ---


,text,keyword,country_code,label,par_id,predicted_label
0,"His present "" chambers "" may be quite humble ,...",homeless,ke,1,107,0
1,Krueger recently harnessed that creativity to ...,disabled,us,1,149,0
2,10:41am - Parents of children who died must ge...,poor-families,in,1,151,0
3,When some people feel causing problem for some...,disabled,ng,1,154,1
4,We are alarmed to learn of your recently circu...,poor-families,ca,1,157,0


## Saving the predictions on official dev set (labeled)

In [11]:
official_dev_df.to_csv("best_model_preds.csv", index=False)

official_dev_df['predicted_label'].to_csv(
    'dev.txt',
    index=False,
    header=False
)

In [12]:
official_dev_df

,text,keyword,country_code,label,par_id,predicted_label
0,"His present "" chambers "" may be quite humble ,...",homeless,ke,1,107,0
1,Krueger recently harnessed that creativity to ...,disabled,us,1,149,0
2,10:41am - Parents of children who died must ge...,poor-families,in,1,151,0
3,When some people feel causing problem for some...,disabled,ng,1,154,1
4,We are alarmed to learn of your recently circu...,poor-families,ca,1,157,0
...,...,...,...,...,...,...
2089,""" The Pakistani police came to our house and t...",refugee,pk,0,10463,0
2090,When Marie O'Donoghue went looking for a speci...,disabled,ie,0,10464,0
2091,Sri Lankan norms and culture inhibit women fro...,women,lk,0,10465,0
2092,He added that the AFP will continue to bank on...,vulnerable,ph,0,10466,0


## Saving the predictions on official test set (unlabeled)

In [13]:
official_test_df = pd.read_csv(
    content_path+"task4_test.tsv",
    sep="\t",
    skiprows=0,
    header=None
)

official_test_df.columns = ["par_id", "art_id", "keyword", "country_code", "text"]

In [14]:
official_test_df

,par_id,art_id,keyword,country_code,text
0,t_0,@@7258997,vulnerable,us,"In the meantime , conservatives are working to..."
1,t_1,@@16397324,women,pk,In most poor households with no education chil...
2,t_2,@@16257812,migrant,ca,The real question is not whether immigration i...
3,t_3,@@3509652,migrant,gb,"In total , the country 's immigrant population..."
4,t_4,@@477506,vulnerable,ca,"Members of the church , which is part of Ken C..."
...,...,...,...,...,...
3827,t_3893,@@20319448,migrant,jm,In a letter dated Thursday to European Commiss...
3828,t_3894,@@9990672,poor-families,au,They discovered that poor families with health...
3829,t_3895,@@37984,migrant,ca,"She married at 19 , to Milan ( Emil ) Badovina..."
3830,t_3896,@@9691377,immigrant,us,The United Kingdom is n't going to devolve int...


In [15]:
official_test_dataset = Dataset.from_pandas(official_test_df)
official_tokenized_test = official_test_dataset.map(tokenize_function, batched=True)

print("\n--- Generating Predictions on Test Set ---")
# predicting on tokenized_test
official_test_outputs = trainer.predict(test_dataset=official_tokenized_test)

# Extract the raw logits
official_raw_logits = official_test_outputs.predictions

# Convert logits to final predicted classes (0 or 1) using argmax
# This looks at the two scores for each row and picks the index of the higher one
official_predicted_classes = np.argmax(official_raw_logits, axis=1)

# Attach the predictions back tp original pandas DataFrame
official_test_df['predicted_label'] = official_predicted_classes

Map: 100%|██████████| 3832/3832 [00:00<00:00, 13051.61 examples/s]



--- Generating Predictions on Test Set ---


In [16]:
official_test_df.head()

,par_id,art_id,keyword,country_code,text,predicted_label
0,t_0,@@7258997,vulnerable,us,"In the meantime , conservatives are working to...",0
1,t_1,@@16397324,women,pk,In most poor households with no education chil...,0
2,t_2,@@16257812,migrant,ca,The real question is not whether immigration i...,0
3,t_3,@@3509652,migrant,gb,"In total , the country 's immigrant population...",0
4,t_4,@@477506,vulnerable,ca,"Members of the church , which is part of Ken C...",0


In [17]:
official_test_df['predicted_label'].to_csv(
    'test.txt',
    index=False,
    header=False
)